In [1]:
# 1. Install system dependencies
!apt-get update && apt-get install -y build-essential cmake git python3-pip

# 2. Clone and Build llama.cpp (The "Engine")
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!mkdir -p build
!cmake -B build
!cmake --build build --config Release -j$(nproc)

# 3. Install Python requirements for the conversion script
!pip install -r requirements.txt
!pip install huggingface_hub

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:13 https://ppa.launchpadcontent.net/ubuntu

In [1]:
import os
# Check if the build worked before the restart
if os.path.exists("/content/llama.cpp/build/bin/llama-quantize"):
    print("chunk-1 is safe")
else:
    print("Something went wrong, need to build again.")

✅ Build is safe! No need to re-run Chunk 1.


In [2]:
# Re-declaring the variables lost during the restart
model_id = 'PY007/TinyLlama-1.1B-Chat-v0.3'
local_dir = "/content/original_model"

print(f"Variables restored. Ready to proceed with {model_id}")

Variables restored. Ready to proceed with PY007/TinyLlama-1.1B-Chat-v0.3


In [3]:
# cleaning
!rm /content/tinyllama-f16.gguf
!rm /content/tinyllama-Q4_K_M.gguf

In [2]:
from huggingface_hub import snapshot_download
import os

model_id = 'PY007/TinyLlama-1.1B-Chat-v0.3'
local_dir = "/content/original_model"

# Download the model weights and config
snapshot_download(
    repo_id=model_id,
    local_dir=local_dir,
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"]
)

print(f"✅ Model downloaded to {local_dir}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

✅ Model downloaded to /content/original_model


In [4]:
# Step A: Convert Safetensors to "Full Precision" GGUF (Format Change)
!python3 /content/llama.cpp/convert_hf_to_gguf.py /content/original_model \
    --outfile /content/tinyllama-f16.gguf \
    --outtype f16



INFO:hf-to-gguf:Loading model: original_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float32 --> F16, shape = {2048, 32003}
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {2048, 32003}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float32 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float32 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.

In [5]:
# Step B: Quantize to 4-bit Q4_K_M (The "Squeeze")
!/content/llama.cpp/build/bin/llama-quantize /content/tinyllama-f16.gguf /content/tinyllama-Q4_K_M.gguf Q4_K_M



main: build = 8356 (b9da4444d)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/tinyllama-f16.gguf' to '/content/tinyllama-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 29 key-value pairs and 201 tensors from /content/tinyllama-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Original_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llam

In [ ]:
!/content/llama.cpp/build/bin/llama-cli \
  -m /content/tinyllama-Q4_K_M.gguf \
  -p "What is the best food in Ahmedabad?" \
  -n 128 \
  --repeat-penalty 1.1

In [7]:
# 1. Install the fast transfer tool
!pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.0 MB/s eta 0:00:00


In [8]:


import os
from huggingface_hub import HfApi

# 2. Enable the turbo mode
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

api = HfApi()

api.upload_file(
    path_or_fileobj="/content/tinyllama-Q4_K_M.gguf",
    path_in_repo="tinyllama-Q4_K_M.gguf",
    repo_id="mahendra4/TinyLlama-1.1B-Chat-v0.3-GGUF",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/tinyllama-Q4_K_M.gguf:   0%|          |  530kB /  668MB            

CommitInfo(commit_url='https://huggingface.co/mahendra4/TinyLlama-1.1B-Chat-v0.3-GGUF/commit/bf93a467943992dfff62062ed9d279d99e6a220c', commit_message='Upload tinyllama-Q4_K_M.gguf with huggingface_hub', commit_description='', oid='bf93a467943992dfff62062ed9d279d99e6a220c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mahendra4/TinyLlama-1.1B-Chat-v0.3-GGUF', endpoint='https://huggingface.co', repo_type='model', repo_id='mahendra4/TinyLlama-1.1B-Chat-v0.3-GGUF'), pr_revision=None, pr_num=None)

In [ ]:
# # other way to push code

# from huggingface_hub import HfApi, login


# login()

# api = HfApi()
# repo_id = "mahendra4/TinyLlama-1.1B-Chat-v0.3-GGUF"

# # 2. Create the repo if it doesn't exist
# api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# # 3. Upload the GGUF file
# api.upload_file(
#     path_or_fileobj="/content/tinyllama-Q4_K_M.gguf",
#     path_in_repo="tinyllama-Q4_K_M.gguf",
#     repo_id=repo_id,
#     repo_type="model"
# )

# print(f"model at: https://huggingface.co/{repo_id}")